In [1]:
import time
import jax
import jax.numpy as jnp
import sparsejac
from jax.experimental import sparse
import phate.tree
import pandas as pd

from uphate.uphate import get_phate_embedding, pdist_squared

ERROR:2025-12-05 11:02:40,918:jax._src.xla_bridge:444: Jax plugin configuration error: Exception when calling jax_plugins.xla_cuda12.initialize()
Traceback (most recent call last):
  File "/home/cpsc4844_tl855/project_cpsc4844/cpsc4844_tl855/uPHATE/.venv/lib64/python3.11/site-packages/jax/_src/xla_bridge.py", line 442, in discover_pjrt_plugins
    plugin_module.initialize()
  File "/home/cpsc4844_tl855/project_cpsc4844/cpsc4844_tl855/uPHATE/.venv/lib64/python3.11/site-packages/jax_plugins/xla_cuda12/__init__.py", line 324, in initialize
    _check_cuda_versions(raise_on_first_error=True)
  File "/home/cpsc4844_tl855/project_cpsc4844/cpsc4844_tl855/uPHATE/.venv/lib64/python3.11/site-packages/jax_plugins/xla_cuda12/__init__.py", line 281, in _check_cuda_versions
    local_device_count = cuda_versions.cuda_device_count()
                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: jaxlib/cuda/versions_helpers.cc:113: operation cuInit(0) failed: CUDA_ERROR_NO_DEVICE


In [2]:
def safe_pdist(x):
    """Compute pairwise distances with numerical stability."""
    eps = 1e-12
    return jnp.sqrt(pdist_squared(x) + eps)


def mds_loss(embedding: jax.Array, data: jax.Array):
    """
    Loss function for MDS, used in custom derivative for MDS solver.
        We pass the key to match the solver signature. Note that jaxopt expects
        the optimality function (in this case grad(loss) == 0) to have the parameters
        we are solving for be the *first* argument."""
    return ((safe_pdist(data) - safe_pdist(embedding)) ** 2).sum() / 2

In [3]:
X = jnp.load("../X.npy")

In [5]:
jnp.isnan(jax.jacobian(jax.grad(mds_loss))(X[:20], X[:20])).any()

Array(False, dtype=bool)

In [236]:
def f(x):
    return jnp.sum(x) + jnp.concatenate((jnp.cumsum(x), jnp.cumsum(x)))


x = jnp.array([0.0, 0.0, 1.0])
y, vjp = jax.vjp(f, x)
J = jax.jacrev(f)(x)
(
    J,
    vjp(jnp.array([1.0, 0.0, 0.0, 0.0, 0.0, 0.0])),
    vjp(jnp.array([0.0, 0.0, 0.0, 0.0, 0.0, 1.0])),
)

(Array([[2., 1., 1.],
        [2., 2., 1.],
        [2., 2., 2.],
        [2., 1., 1.],
        [2., 2., 1.],
        [2., 2., 2.]], dtype=float32),
 (Array([2., 1., 1.], dtype=float32),),
 (Array([2., 2., 2.], dtype=float32),))

In [243]:
sparsity = sparse.BCOO.fromdense(jax.random.bernoulli(jax.random.PRNGKey(0), p=0.5, shape=J.shape))
sparse_J = sparsejac.jacrev(f, sparsity=sparsity)(x)
sparse_J.todense()

Array([[0., 0., 3.],
       [6., 0., 4.],
       [4., 0., 0.],
       [2., 0., 1.],
       [0., 0., 0.],
       [0., 0., 0.]], dtype=float32)

In [53]:
key = jax.random.PRNGKey(0)
n_samples, n_features = 100, 10
X = jax.random.normal(key, (n_samples, n_features))

print(
    f"Benchmarking Jacobian for N={n_samples}, D={n_features}"
)

# Warmup
print("Warmup...")
_ = get_phate_embedding(X, key, t=2, n_components=2)

# Define function to differentiate
def embedding_fn(x):
    return get_phate_embedding(x, key, t=2, n_components=2)

# Measure time
start_time = time.time()
J = jax.jit(jax.jacrev(embedding_fn))(X)
# Block until ready
J.block_until_ready()
end_time = time.time()

print(
    f"Jacobian shape: {J.shape}, Time taken: {end_time - start_time:.4f} seconds"
)

Benchmarking Jacobian for N=100, D=10
Warmup...
Jacobian shape: (100, 2, 100, 10), Time taken: 15.0306 seconds


In [ ]:
@jax.jit
def sparse_jacrev_fn(x):
  with jax.ensure_compile_time_eval():
    sparsity = sparse.BCOO.fromdense(jnp.eye(10000))
    jacrev_fn = sparsejac.jacrev(fn, sparsity=sparsity)
  return jacrev_fn(x)

In [ ]:
def knn_indices(X, k):
    pairwise_dist = pdist_squared(X)
    indices = jax.lax.top_k(-pairwise_dist, k=k + 1)[1]
    return indices

X = jax.random.normal(key, (n_samples, n_features))
indices = knn_indices(X, k=5)

In [ ]:
k = 5

pairwise_dist = pdist_squared(X)
values = -jax.lax.top_k(-pairwise_dist, k=5)[0][:, -1]
indices = jax.lax.top_k(-pairwise_dist, k=k + 1)[1]
sparse_pattern = sparse.BCOO.fromdense(jnp.where(pairwise_dist <= values[:, None], 1., 0.))